<a href="https://colab.research.google.com/github/rburchf1/AI102Challenges/blob/main/ai301_assignment1_keepemojis_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#Install/Import basics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


print('OK: libraries imported')

OK: libraries imported


In [3]:
# Load Twitter Dataset
file_path = '/content/drive/MyDrive/Tweets.csv'
try:
  ds = pd.read_csv(file_path)
  print('File loaded successfully.')
except FileNotFoundError:
  print(f'File not found at path: {file_path}. Please check the file path.')
except Exception as e:
  print(f'An error occurred while loading the file: {e}')

#Generate basic information about dataset
print("DataSet Info:")
ds.info()

#Create dataset description
print("\nDataSet Description:")
display(ds.describe(include='all'))

#View a few rows from the dataset
print("\nFirst 5 rows of the DataSet:")
display(ds.head())

File loaded successfully.
DataSet Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created       

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
count,1.464000e+04,14640,14640.000000,9178,10522.000000,14640,40,14640,32,14640.000000,14640,1019,14640,9907,9820
unique,NaN,3,NaN,10,NaN,6,3,7701,13,NaN,14427,832,14247,3081,85
top,NaN,negative,NaN,Customer Service Issue,NaN,United,negative,JetBlueNews,Customer Service Issue,NaN,@united thanks,"[0.0, 0.0]",2015-02-24 09:54:34 -0800,"Boston, MA",Eastern Time (US & Canada)
freq,NaN,9178,NaN,2910,NaN,3822,32,63,12,NaN,6,164,5,157,3744
mean,5.692184e+17,NaN,0.900169,NaN,0.638298,NaN,NaN,NaN,NaN,0.082650,NaN,NaN,NaN,NaN,NaN
std,7.791112e+14,NaN,0.162830,NaN,0.330440,NaN,NaN,NaN,NaN,0.745778,NaN,NaN,NaN,NaN,NaN
min,5.675883e+17,NaN,0.335000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
25%,5.685592e+17,NaN,0.692300,NaN,0.360600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
50%,5.694779e+17,NaN,1.000000,NaN,0.670600,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
75%,5.698905e+17,NaN,1.000000,NaN,1.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN



First 5 rows of the DataSet:


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [4]:
# Split data for training and validation and testing
from sklearn.model_selection import train_test_split

x = ds['text']
y = ds['airline_sentiment']

#FIRST SPLIT
#80% training + validation, 20% for test

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#SECOND SPLIT
#Take 12.5% of the 80% for validation
#12.5% of 80% = 10% of the original dataset

x_train, x_val, y_train, y_val = train_test_split(
    x, y,
    test_size=0.125,
    random_state=42,
    stratify=y
)

In [5]:
#EMOJIS
#Detect if there are emojies in the dataset using Regex.
#Note: This section of code was generated using the Gemini integration in Google Colab

import re

emoji_pattern = re.compile(
    "["  # Start character set
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
    "\U0001F680-\U0001F6FF"  # Transport and Map Symbols
    "\U0001F1E0-\U0001F1FF"  # Regional indicator symbols
    "\U00002600-\U000026FF"  # Miscellaneous Symbols
    "\U00002700-\U000027BF"  # Dingbats
    "]+"
)

def contains_emoji(text):
    return bool(emoji_pattern.search(str(text)))

# Originally sampled some texts to check for emojis but no emojis were found.
# Given the details of the assignment I expected emojis, so I needed to check the whole dataset.
# The sample check was commented out (below).
#print("Checking for emojis in sample texts:")
#found_emoji = False
#for i, text in enumerate(x.sample(n=10, random_state=42)):
    #if contains_emoji(text):
        #print(f"  Text {i+1} (contains emoji): {text}")
        #found_emoji = True
    #else:
        #print(f"  Text {i+1} (no emoji): {text}")

#if not found_emoji:
    #print("  No emojis found in the selected sample of texts.")
#else:
    #print("  Emojis were found in the sample texts.")

#print("\nNow checking the full dataset (this might take a moment if the dataset is large):")

# Check the entire 'text' column for emojis
num_texts_with_emojis = x.apply(contains_emoji).sum()

if num_texts_with_emojis > 0:
    print(f"The 'text' column contains emojis. Found {num_texts_with_emojis} texts with emojis.")
    print("It would be beneficial to decide whether to remove or preserve them during preprocessing based on the task.")
else:
    print("The 'text' column does not appear to contain emojis.")

The 'text' column contains emojis. Found 487 texts with emojis.
It would be beneficial to decide whether to remove or preserve them during preprocessing based on the task.


In [6]:
#Define and apply a clean function

def basic_clean(text):
    text = str(text).lower()  # Convert to string and lowercase

    # Remove URLs (http/https links)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove mentions (@usernames)
    text = re.sub(r'@\w+', '', text)

    # Commented out the removal of emojies
    # Remove emojis using the previously defined pattern
    # text = emoji_pattern.sub(r'', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function
clean_x_train = x_train.apply(basic_clean)
clean_x_val = x_val.apply(basic_clean)
clean_x_test = x_test.apply(basic_clean)

print("Original text sample:", x_train.iloc[0])
print("Cleaned text sample:", clean_x_train.iloc[0])

print("\nFirst 5 cleaned training texts:")
display(clean_x_train.head())

Original text sample: @VirginAmerica Can't bring up my reservation online using Flight Booking Problems code
Cleaned text sample: can't bring up my reservation online using flight booking problems code

First 5 cleaned training texts:


,text
86,can't bring up my reservation online using fli...
14047,educate bohol is a 501(c)(3) w/all volunteer s...
3642,i mean is there a real live person somewhere i...
2356,how about plowing the snow at a gate before th...
5455,i met my twitter friend waiting outside the tr...


In [9]:
#Additional preprocessing: tokenization, remove stopwords, remove puncutation, stemming, and lemmatization

#Reference: This section of code was adapted from AI102 (NLP Techniques) Module 3 Guided Lab.

import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
import string

nltk.download('stopwords')
nltk.download('wordnet')

text = "Natural Language Processing enables computers to understand human language."

ext = text.translate(str.maketrans('', '', string.punctuation))

# Tokenize (split) the text into separate words.
# text.split() breaks the sentence into a list of words using spaces.
tokens = text.split()

# Get the set of English stop words.
# A set is used because it allows fast checking of "is this word in the list?"
stop_words = set(stopwords.words('english'))

# Remove stop words from our list of tokens.
# This list comprehension means:
# "for each word in tokens, keep it only if it is NOT in stop_words".
filtered_tokens = [word for word in tokens if word not in stop_words]

# Create (initialize) the stemmer and lemmatizer objects that we will use.
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Apply stemming and then lemmatization to each filtered word.
# For each word in filtered_tokens:
#   1. stemmer.stem(word) gets the stem (basic form)
#   2. lemmatizer.lemmatize(...) changes that stem to a dictionary form if possible
# The final results are stored in a new list called processed_tokens.
processed_tokens = [lemmatizer.lemmatize(stemmer.stem(word)) for word in filtered_tokens]

# Print the cleaned version of the text (lowercased and without punctuation)
print("Original Text:")
print(text)

# Print a blank line and then the tokens after removing stop words
print("\nFiltered Tokens (Stop Words Removed):")
print(filtered_tokens)

# Print another blank line and then the tokens after stemming and lemmatization
print("\nProcessed Tokens (Stemmed and Lemmatized):")
print(processed_tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Original Text:
Natural Language Processing enables computers to understand human language.

Filtered Tokens (Stop Words Removed):
['Natural', 'Language', 'Processing', 'enables', 'computers', 'understand', 'human', 'language.']

Processed Tokens (Stemmed and Lemmatized):
['natur', 'languag', 'process', 'enabl', 'comput', 'understand', 'human', 'language.']


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1,2))
xtr = tfidf.fit_transform(x_train)
xva = tfidf.transform(x_val)
clf = LogisticRegression(max_iter=200)
clf.fit(xtr, y_train)
preds = clf.predict(xva)

print(classification_report(y_val, preds, digits=4))

              precision    recall  f1-score   support

    negative     0.8166    0.9590    0.8821      1147
     neutral     0.7182    0.5387    0.6156       388
    positive     0.8333    0.5424    0.6571       295

    accuracy                         0.8027      1830
   macro avg     0.7894    0.6800    0.7183      1830
weighted avg     0.7985    0.8027    0.7893      1830



In [8]:
emoji_features = [term for term in tfidf.get_feature_names_out()
                  if any(ord(char) > 10000 for char in term)]

print(emoji_features[:20])
print(len(emoji_features))

[]
0


The output 0 indicate that no emoji features were identified in the vocabulary generated by the TfidfVectorizer. This suggests that any emojis present in the original text (num_texts_with_emojis = 487 was previously found) were either removed during the basic_clean function or were not captured by the TfidfVectorizer due to its max_features or other internal processing.